# Eval Harness — Tool-Use Specialist

Compares models on **100 held-out tool-calling examples** (same seed=42 across all runners):
- Fine-tuned Gemma 2 2B (`Sahil3717/gemma-2-2b-tool-use-lora`)
- Base Gemma 2 2B (`google/gemma-2-2b-it`)
- Llama 3.3 70B via Groq (~73k tokens, fits 100k/day free limit)
- Gemini 2.0 Flash via AI Studio *(skipped automatically if not available in your region)*

**Before running:** Set Accelerator → GPU T4 x1. Add Kaggle secrets: `HF_TOKEN`, `GROQ_API_KEY`, `GEMINI_API_KEY`.

Run cells **top to bottom**. Each runner caches its JSON — re-running after a crash resumes from where it left off. If you need to re-run a specific runner from scratch, delete its file in `eval/results/` first.

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'GPU: {result.stdout.strip()}')
else:
    print('No GPU — go to Settings -> Accelerator -> GPU T4 x1')

In [ ]:
%%capture
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.31.0 \
    datasets>=2.19.0 \
    huggingface-hub>=0.23.0 \
    groq>=0.9.0 \
    google-genai>=1.0.0 \
    sentencepiece>=0.2.0 \
    protobuf>=5.27.0 \
    python-dotenv>=1.0.0

import transformers
print(f'transformers {transformers.__version__}')
print('All dependencies installed')

In [ ]:
import os

if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ['HF_TOKEN']       = s.get_secret('HF_TOKEN')
    os.environ['GROQ_API_KEY']   = s.get_secret('GROQ_API_KEY')
    os.environ['GEMINI_API_KEY'] = s.get_secret('GEMINI_API_KEY')
    print('Running on Kaggle')
else:
    from google.colab import userdata
    os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
    os.environ['GROQ_API_KEY']   = userdata.get('GROQ_API_KEY')
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('Running on Colab')

for key in ['HF_TOKEN', 'GROQ_API_KEY', 'GEMINI_API_KEY']:
    print(f'{"OK" if os.environ.get(key) else "MISSING"} {key}')

In [ ]:
import os
from pathlib import Path

REPO     = 'function-call-finetune'
REPO_URL = 'https://github.com/sahilmdeshmukh/function-call-finetune.git'

if Path(REPO).exists():
    print('Repo exists — pulling...')
    !git -C {REPO} pull
else:
    !git clone {REPO_URL}

os.chdir(REPO)
print(f'Working dir: {os.getcwd()}')

In [ ]:
import os
from pathlib import Path

if Path('data/test_held_out.jsonl').exists():
    print('data/test_held_out.jsonl already exists — skipping prepare')
else:
    print('Generating data splits from Salesforce/xlam-function-calling-60k ...')
    token = os.environ.get('HF_TOKEN', '')
    !HF_TOKEN={token} python data/prepare.py
    print('Data splits ready')

In [ ]:
# OPTIONAL — only run this cell if you want to clear cached results and re-run from scratch.
# Leave it unrun during normal execution.
import os
for f in [
    'eval/results/results_groq.json',
    'eval/results/results_gemini.json',
    'eval/results/results_base.json',
    'eval/results/results_finetuned.json',
]:
    if os.path.exists(f):
        os.remove(f)
        print(f'Cleared {f}')
print('Done — re-run the runner cells below')

In [ ]:
from eval.groq_runner import run as groq_run

groq_results = groq_run(
    test_path='data/test_held_out.jsonl',
    n_samples=100,
    output_path='eval/results/results_groq.json',
)
print(f'Groq: {len(groq_results)} examples done')

In [ ]:
try:
    from eval.gemini_runner import run as gemini_run
    gemini_results = gemini_run(
        test_path='data/test_held_out.jsonl',
        n_samples=100,
        output_path='eval/results/results_gemini.json',
    )
    print(f'Gemini: {len(gemini_results)} examples done')
except Exception as e:
    print(f'Gemini skipped: {e}')
    print('Continuing without Gemini — report will show 3 models')

In [ ]:
from eval.model_runner import run as model_run

base_results, ft_results = model_run(
    test_path='data/test_held_out.jsonl',
    n_samples=100,
    base_output_path='eval/results/results_base.json',
    finetuned_output_path='eval/results/results_finetuned.json',
)
print(f'Base: {len(base_results)} | Fine-tuned: {len(ft_results)} examples done')

In [ ]:
from eval.report import generate

generate(
    results_dir='eval/results',
    output_md='eval/results/results.md',
)

In [ ]:
!git add eval/results/results.md
!git commit -m 'feat: add eval results -- Day 3 complete'
!git push
print('Results committed and pushed!')